# CVD_Active goal: fixed-question contextual Fusion answers

Select a process occurrence, its Step and an analysis-center sensor separately.
Keep the existing sensor-free English questions. Current/reference traces and all
context sensors enter Fusion; a trained contextual head returns answer logits.
q99/GMM are explicit numerical comparators, never fallback answers. Historical
outputs below predate this goal until rerun. No real fault quality is established.


양방향·가변 구간 학습 보강: 기본 주입에 증가/감소와 무작위 위치·길이를 추가합니다.
ORDINAL은 구간 내부의 합성 주입 진폭 단계이며 지속시간·실제 고장 심각도가 아닙니다.
STEP 1·2는 재사용하고 STEP 3 답변 헤드만 새 소스 identity 경로로 학습합니다.
기존 답변 가중치는 보존됩니다. 사례 수가 늘어 특징 준비 시간이 증가합니다.


# STEP3 - Train contextual Fusion answer heads

Frozen STEP1 encoders and STEP2 Fusion produce selected-center features.
Only TRAIN synthetic examples fit the readout; Validation selects epochs.
BOOLEAN: changed selected sensor. CHOICE: change family including Delay.
ORDINAL: local amplitude stages (.5, 1, 2, 4), or native Delay lags (1, 2, 3, 4).
These are different synthetic units, not physical fault severity.
Controls use exact self-reference and off-center changes. Original cross-run
normality is not established by those controls. No numerical gate chooses answers.
Existing compatible weights are reused. Source/config changes create a new identity.
Previously inspected Test is a development benchmark; it is never used by this
training function, but past Test-guided changes prevent a final-holdout claim.


In [ ]:
# Shared settings: edit config/experiments/gevis_pipeline.json, then restart kernels.
from pathlib import Path
import sys, os
PIPELINE_ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir())
if str(PIPELINE_ROOT/'src') not in sys.path: sys.path.insert(0,str(PIPELINE_ROOT/'src'))
import importlib
import pecvd.experiments.pipeline_settings as pipeline_settings
pipeline_settings=importlib.reload(pipeline_settings)
import importlib
importlib.reload(importlib.import_module("pecvd.experiments.questions"))
importlib.reload(importlib.import_module("pecvd.model_variants"))
PIPELINE_SETTINGS=pipeline_settings.configure_pipeline(PIPELINE_ROOT)
print({k:PIPELINE_SETTINGS[k] for k in ('model_variant','training_profile','backbone_revision','decision_system') if k in PIPELINE_SETTINGS})

from pecvd.experiments.questions import PRIMARY_GOAL_VERSION
from pecvd.experiments.fusion_answers import SYSTEM as GEVIS_SYSTEM, KINDS, QUESTIONS
if PIPELINE_SETTINGS.get('decision_system')!='fusion_answers':
    raise RuntimeError('Active notebooks require decision_system=fusion_answers.')
print({'question_contract':PRIMARY_GOAL_VERSION,'gevis_system':GEVIS_SYSTEM,'decision_path':'learned Fusion readout',
       'questions':{k:QUESTIONS[k] for k in KINDS},'answer_types':KINDS,
       'training':'STEP3 TRAIN readout; Validation selection; frozen STEP1/2',
       'interval_mode':'unavailable in this answer path',
       'probabilities':'softmax scores; not calibrated fault probabilities'})


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter, defaultdict
import copy, hashlib, importlib, json, math, random, sys, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir()),None)
if ROOT is None: raise RuntimeError('Run inside the PECVD project')
sys.path.insert(0,str(ROOT/'src'))
from pecvd.process_config import load_process_config
import pecvd.gevis_fast as gevis_fast
if Path(gevis_fast.__file__).resolve()!=(ROOT/'src'/'pecvd'/'gevis_fast.py').resolve(): raise RuntimeError('GEVIS helper comes from another workspace; restart the kernel in this project')
gevis_fast=importlib.reload(gevis_fast)  # Refresh helper definitions in long-lived notebook kernels
if not all(hasattr(gevis_fast,name) for name in ('pack_boolean_records','configure_boolean_fusion','boolean_fusion_logits','BooleanNumericProbe','evaluate_boolean','fit_boolean','typed_targets','pack_typed_records','fit_typed_comparison','choose_boolean_threshold')): raise RuntimeError('GEVIS diagnostic helper is outdated on disk; reload the updated project files')
from pecvd.gevis_fast import frozen_text, configure_fast, fit_heads
import pecvd.gevis_backbone as gevis_backbone
gevis_backbone=importlib.reload(gevis_backbone)
render_semantic_question=gevis_backbone.render_semantic_question
BACKBONE_QUESTION_BANK=gevis_backbone.QUESTION_BANK
BACKBONE_SOURCE_SCHEMA=gevis_backbone.BACKBONE_SOURCE_SCHEMA
BACKBONE_SOURCE_SHA256=hashlib.sha256(Path(gevis_backbone.__file__).resolve().read_bytes()).hexdigest()
from pecvd.experiments.process_inputs import configured_process_path,candidate_backbone_path
PROCESS_CONFIG_PATH=configured_process_path(ROOT)
import importlib
import pecvd.model_variants as model_variants
importlib.reload(model_variants)
from pecvd.model_variants import active_config,active_run_tag,selected_variant
PROCESS_CONFIG=active_config(load_process_config(PROCESS_CONFIG_PATH))
DATA_PATH=ROOT/PROCESS_CONFIG.data_path
ENCODER_ARTIFACT_VERSION=PROCESS_CONFIG.artifacts.encoder
BACKBONE_ARTIFACT_VERSION='gevis_backbone_e5_v1'
BACKBONE_CURRICULUM_TAG='fdc_semantic_step1_bridge_all_types_v1'
BACKBONE_RUN_TAG=active_run_tag()
SESSION_SIGNATURE='STEP3_GEVIS_PRETRAINED_POLICY_V2'
BACKBONE_EXPERIMENT_TAG=BACKBONE_CURRICULUM_TAG+'_'+BACKBONE_RUN_TAG
MODEL_ARTIFACT_VERSION=PROCESS_CONFIG.artifacts.decision
ENCODER_ROOT=ROOT/'artifacts'/ENCODER_ARTIFACT_VERSION
BACKBONE_ROOT=candidate_backbone_path(ROOT,PROCESS_CONFIG,BACKBONE_RUN_TAG).parent
MODEL_ROOT=ROOT/'artifacts'/MODEL_ARTIFACT_VERSION
RUN_COLUMN=PROCESS_CONFIG.columns.run; STEP_COLUMN=PROCESS_CONFIG.columns.step; ORDER_COLUMN=PROCESS_CONFIG.columns.order
STEPS=PROCESS_CONFIG.step_specs

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def save_torch(obj,path,overwrite=False):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and not overwrite: raise FileExistsError(f'{path} exists; set OVERWRITE_ARTIFACTS=True intentionally')
    tmp=path.with_suffix(path.suffix+'.tmp'); torch.save(obj,tmp); tmp.replace(path)

def save_json(obj,path,overwrite=False):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and not overwrite: raise FileExistsError(f'{path} exists; set OVERWRITE_ARTIFACTS=True intentionally')
    tmp=path.with_suffix(path.suffix+'.tmp'); tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8'); tmp.replace(path)

def derive_step_label(frame):
    if STEP_COLUMN in frame: return frame
    prefix=PROCESS_CONFIG.columns.step_one_hot_prefix; excluded=set(PROCESS_CONFIG.columns.step_one_hot_excluded)
    cols=[x for x in frame if x.startswith(prefix) and x not in excluded]
    if not cols: raise KeyError('step_label and step_name_* are both missing')
    values=frame[cols].fillna(0).to_numpy(); ix=values.argmax(1); strength=values.max(1)
    out=frame.copy(); labels=np.array([x.removeprefix(prefix) for x in cols],object)[ix]
    out[STEP_COLUMN]=np.where(strength>0,labels,None); return out

def select_sensor_columns(frame):
    from pecvd.experiments.process_inputs import configured_sensors
    explicit=configured_sensors(PROCESS_CONFIG,frame)
    if explicit is not None: return explicit
    forbidden={RUN_COLUMN,STEP_COLUMN,ORDER_COLUMN,'row_in_step','local_sample_id','global_sample_id','local_process_id','global_process_id','sample_order','sample_order_within_timestamp','timestamp_group_size','exact_value_group_size','is_exact_duplicate','step_elapsed_seconds','step_row_count','elapsed_seconds','elapsed_from_declared_start_seconds','elapsed_from_first_sample_seconds','data_split','step_segment_id','step_number','step_progress','previous_step_label','previous_step_number','previous_step_duration','previous_step_row_count','previous_step_end_order'}
    prefixes=('step_name_','source_group_','previous_step_','pm_cycle_','recipe_','label_','target_')
    cols=[x for x in frame.select_dtypes(include=np.number) if x not in forbidden and not x.startswith(prefixes)]
    if not cols: raise ValueError('No physical numeric sensor columns remain')
    return cols

def split_run_ids(frame,seed=SEED):
    ids=np.array(sorted(frame[RUN_COLUMN].dropna().unique().tolist()),dtype=object); np.random.default_rng(seed).shuffle(ids)
    n=len(ids); a=max(1,int(.70*n)); b=max(a+1,a+int(.15*n))
    result={'train':ids[:a].tolist(),'validation':ids[a:b].tolist(),'test':ids[b:].tolist()}
    sets={k:set(v) for k,v in result.items()}; assert not sets['train']&sets['validation'] and not sets['train']&sets['test'] and not sets['validation']&sets['test']
    return result

class PatchEncoder(nn.Module):
    def __init__(self,n_sensors,patch_len,d_model=64,nhead=4,layers=2,dropout=.1):
        super().__init__(); self.n_sensors=n_sensors; self.patch_len=patch_len; self.d_model=d_model
        self.input=nn.Linear(patch_len,d_model); self.sensor_emb=nn.Embedding(n_sensors,d_model); self.patch_emb=nn.Embedding(64,d_model); self.mask_token=nn.Parameter(torch.zeros(d_model))
        block=nn.TransformerEncoderLayer(d_model,nhead,4*d_model,dropout,batch_first=True,norm_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(block,layers); self.norm=nn.LayerNorm(d_model); self.decoder=nn.Linear(d_model,patch_len)
    def patchify(self,x):
        b,t,s=x.shape; p=t//self.patch_len
        return x[:,:p*self.patch_len].reshape(b,p,self.patch_len,s).permute(0,3,1,2).reshape(b,s*p,self.patch_len),p
    def tokens(self,x,mask=None):
        z,p=self.patchify(x); b=z.shape[0]; sid=torch.arange(self.n_sensors,device=x.device).repeat_interleave(p); pid=torch.arange(p,device=x.device).repeat(self.n_sensors)
        content=self.input(z)
        if mask is not None: content=torch.where(mask[...,None],self.mask_token[None,None],content)
        h=content+self.sensor_emb(sid)[None]+self.patch_emb(pid)[None]
        return self.norm(self.encoder(h))
    def forward(self,x,mask=None): return self.decoder(self.tokens(x,mask))

def deterministic_mask(batch,tokens,ratio,seed,device):
    g=torch.Generator(device=device); g.manual_seed(seed)
    mask=torch.rand((batch,tokens),generator=g,device=device)<ratio; empty=~mask.any(dim=1)
    if empty.any():
        rows=empty.nonzero(as_tuple=True)[0]; fallback=torch.randint(0,tokens,(len(rows),),generator=g,device=device); mask[rows,fallback]=True
    return mask

def masked_metrics(model,x,ratio=.25,seed=0):
    target,_=model.patchify(x); mask=deterministic_mask(len(x),target.shape[1],ratio,seed,x.device)
    pred=model(x,mask); den=mask.sum().clamp_min(1)*target.shape[-1]
    mse=((pred-target).pow(2)*mask[...,None]).sum()/den
    zero=(target.pow(2)*mask[...,None]).sum()/den
    return mse,zero,mask,pred,target

ANSWER_TYPES={'BOOLEAN':0,'CHOICE':1,'ORDINAL':2}
class ByteTextEncoder(nn.Module):
    def __init__(self,d_model=128,max_bytes=320,layers=2,dropout=.1):
        super().__init__(); self.max_bytes=max_bytes; self.token=nn.Embedding(257,d_model,padding_idx=0); self.position=nn.Embedding(max_bytes,d_model)
        block=nn.TransformerEncoderLayer(d_model,4,4*d_model,dropout,batch_first=True,norm_first=True,activation='gelu')
        self.encoder=nn.TransformerEncoder(block,layers); self.norm=nn.LayerNorm(d_model)
    def forward(self,texts,device):
        encoded=[list(text.encode('utf-8')[:self.max_bytes]) for text in texts]; width=max(1,max(map(len,encoded)))
        ids=torch.zeros((len(texts),width),dtype=torch.long,device=device)
        for i,raw in enumerate(encoded):
            if raw: ids[i,:len(raw)]=torch.tensor([x+1 for x in raw],device=device)
        mask=ids.eq(0); pos=torch.arange(width,device=device)[None]
        z=self.encoder(self.token(ids)+self.position(pos),src_key_padding_mask=mask)
        keep=(~mask).unsqueeze(-1); return self.norm((z*keep).sum(1)/keep.sum(1).clamp_min(1))

class GEVIS(nn.Module):
    def __init__(self,numeric_dim=256,d_model=128,dropout=.1,text_encoder_name='byte',text_encoder_revision=None):
        super().__init__(); self.text_encoder_name=text_encoder_name; self.text_encoder_revision=text_encoder_revision; self.text=ByteTextEncoder(d_model,dropout=dropout) if text_encoder_name=='byte' else gevis_backbone.E5TextEncoder(d_model,model_name=text_encoder_name,revision=text_encoder_revision or gevis_backbone.DEFAULT_TEXT_ENCODER_REVISION); self.numeric_input_norm=nn.LayerNorm(numeric_dim); self.numeric=nn.Linear(numeric_dim,d_model); self.numeric_key_norm=nn.LayerNorm(d_model); self.descriptor_norm=nn.LayerNorm(d_model); self.fusion_gate=nn.Sequential(nn.Linear(2*d_model,d_model),nn.Sigmoid()); self.type_emb=nn.Embedding(3,d_model)
        from pecvd.temporal_bridge import install
        install(self,numeric_dim,d_model)
        self.cross=nn.MultiheadAttention(d_model,4,dropout,batch_first=True); self.answer_norm=nn.LayerNorm(d_model); self.support_head=nn.Sequential(nn.LayerNorm(d_model),nn.Linear(d_model,64),nn.GELU(),nn.Linear(64,1)); self.change_head=nn.Sequential(nn.LayerNorm(2*d_model),nn.Linear(2*d_model,d_model),nn.GELU(),nn.Dropout(dropout),nn.Linear(d_model,1))
        self.semantic_heads=nn.ModuleDict({name:nn.Linear(d_model,size) for name,size in {'statistic':5,'direction':3,'time_scope':3}.items()})
        self.criterion_proj=nn.Sequential(nn.Linear(d_model,d_model),nn.GELU(),nn.LayerNorm(d_model)); self.temperature=nn.Parameter(torch.tensor(0.0))
    def load_state_dict(self,*args,**kwargs):
        self._text_cache=None
        return super().load_state_dict(*args,**kwargs)
    def _text(self,texts,device): return frozen_text(self,texts,device)
    def support_probability(self,questions): return self.support_head(self._text(questions,next(self.parameters()).device)).squeeze(-1).sigmoid()
    def forward(self,numeric_tokens,questions,answer_types,criteria,token_descriptions=None,return_fusion=False):
        device=numeric_tokens.device; type_ids=torch.tensor([ANSWER_TYPES[x] for x in answer_types],device=device)
        if numeric_tokens.shape[-1]!=self.numeric.in_features: raise ValueError(f'numeric state width {numeric_tokens.shape[-1]} does not match GEVIS input width {self.numeric.in_features}; rerun STEP 2 and STEP 3 with the current rich-SSL contract')
        text_query=self._text(questions,device); q=text_query+self.type_emb(type_ids); numeric_keys=self.numeric_key_norm(self.numeric(self.numeric_input_norm(numeric_tokens))); descriptors=None
        if token_descriptions is not None:
            if len(token_descriptions)!=len(numeric_tokens) or any(len(items)!=numeric_tokens.shape[1] for items in token_descriptions): raise ValueError('token descriptor shape mismatch')
            flat=[text for items in token_descriptions for text in items]
            unique=list(dict.fromkeys(flat)); lookup={text:index for index,text in enumerate(unique)}
            indices=torch.tensor([lookup[text] for text in flat],device=device,dtype=torch.long)
            # Reuse within this forward only: index_select preserves gradients; no stale training cache.
            descriptors=self.descriptor_norm(self._text(unique,device).index_select(0,indices).reshape_as(numeric_keys))
        from pecvd.gevis_backbone import attention_inputs
        attention_query,keys=attention_inputs(numeric_keys,descriptors,text_query,q,self.fusion_gate,getattr(self,'fusion_mode','legacy'))
        answer,attention=self.cross(attention_query[:,None],keys,keys,need_weights=True); answer=self.answer_norm(answer[:,0]+q)
        fusion=(answer,q,attention[:,0])
        if return_fusion: return fusion
        return self.decode(fusion,answer_types,criteria),attention[:,0]
    def decode(self,fusion,answer_types,criteria):
        answer,q,_=fusion; device=answer.device
        change_logit=self.change_head(torch.cat([answer,q],-1)).squeeze(-1)
        scale=self.temperature.exp().clamp(1e-2,100)
        flat=[item for items in criteria for item in items]; all_emb=self.criterion_proj(self._text(flat,device)); logits=[]; offset=0
        for i,items in enumerate(criteria):
            emb=all_emb[offset:offset+len(items)]; offset+=len(items); score=scale*(emb@answer[i])/math.sqrt(answer.shape[-1])
            if answer_types[i]=='BOOLEAN' and len(items)==2: score=score+torch.stack([-change_logit[i],change_logit[i]])
            logits.append(score)
        return logits

PAIR_PROTOCOL='real_cross_run_grounded_question_rewards'
LJR_NAME='Learning from Joint Rewards'


@dataclass(frozen=True)
class GevisTrainConfig:
    epochs:int=8
    batch_size:int=32
    learning_rate:float=1e-3
    min_learning_rate:float=2e-5
    weight_decay:float=1e-4
    patience:int=2
    scheduler_patience:int=2
    evidence_reward_weight:float=.20
    consistency_loss_weight:float=.35
    d_model:int=192
    dropout:float=.1
    text_encoder_name:str=gevis_backbone.DEFAULT_TEXT_ENCODER
    text_encoder_revision:str=gevis_backbone.DEFAULT_TEXT_ENCODER_REVISION
    max_windows_per_step:int=128
    max_validation_windows_per_step:int=64
    max_fit_seconds:float=120.
    max_cache_mib:float=128.
    adaptation_mode:str='cached_heads'
    gaussian_watch_z:float=2.0
    gaussian_alert_z:float=3.0
    gaussian_min_reference_windows:int=20
    ljr_reward_weight:float=.30
    reward_temperature:float=.75
    entropy_weight:float=.01
    overwrite_artifacts:bool=False

CONFIG=GevisTrainConfig()
# Calibration/evaluation only. Historical head-fitting switches are disabled.
DECISION_SYSTEM=globals().get('PIPELINE_SETTINGS',{}).get('decision_system','hybrid')
RUN_FUSION_ANSWER_TRAINING=DECISION_SYSTEM=='fusion_answers'
RUN_STRUCTURED_EVALUATION=not RUN_FUSION_ANSWER_TRAINING
RUN_TRAINING=False  # Never retrain STEP1/2 here.
RUN_ANSWER_HEAD_TRAINING=False  # Optional ablation ONLY; never the primary GEVIS detector.
RUN_LEARNED_CENTER_TRAINING=False
RUN_TEMPORAL_LOCALIZATION_TRAINING=not RUN_FUSION_ANSWER_TRAINING  # Interval-only head; detection is unchanged.
RUN_INTEGRATED_EXPERIMENT=False
RUN_BOOLEAN_DIAGNOSTIC=False
TEXT_ENCODER_ONLY=False
QUESTION_BANK={split:{answer_type:[item.text for item in items if item.supported and item.answer_type==answer_type] for answer_type in ANSWER_TYPES} for split,items in BACKBONE_QUESTION_BANK.items()}
# Named-sensor QUESTION_BANK is diagnostic-only. Automatic GEVIS discovery uses these sensor-free prompts.
from pecvd.experiments.questions import DISCOVERY_QUESTIONS
DISCOVERY_QUESTION_BANK=dict(DISCOVERY_QUESTIONS)
CRITERIA={k:list(v) for k,v in gevis_fast.DEFAULT_ANSWER_CRITERIA.items()}


In [ ]:
manifest_path=ENCODER_ROOT/'encoder_manifest.json'
if not manifest_path.exists(): raise FileNotFoundError('Run notebook 01 for gevis_v4 first')
manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
if manifest.get('schema')!='gevis.encoder_manifest.rich_ssl.v1': raise ValueError('Expected rich-SSL encoder manifest v1')
if set(manifest['step_checkpoints'])!=set(STEPS): raise ValueError('Configured Steps do not match the encoder artifact')
for step,item in manifest['step_checkpoints'].items():
    expected={'window_length':STEPS[step][0],'patch_length':STEPS[step][1],'stride':STEPS[step][2]}
    if any(item['step_config'][key]!=value for key,value in expected.items()): raise ValueError(f'Configured Step shape does not match encoder artifact: {step}')
encoders={}; windows={}; provenance={}
for step,item in manifest['step_checkpoints'].items():
    path=ENCODER_ROOT/item['path']
    if sha256_file(path)!=item['sha256']: raise ValueError(f'Encoder hash mismatch: {step}')
    ck=torch.load(path,map_location='cpu',weights_only=False)
    if ck.get('schema')!='gevis.encoder.rich_ssl.v1': raise ValueError(f'Unsupported encoder: {step}')
    from pecvd.encoder_loading import load_encoder  # Refresh local factories before strict offline reconstruction.
    cfg=ck['model_config']; model=load_encoder(ck,PatchEncoder); model.eval(); encoders[step]=(model,ck)
    with np.load(ENCODER_ROOT/'windows'/f'{step}.npz') as z:
        windows[step]={split:z[split] for split in ('train','validation','test')}
        provenance[step]={split:{column:z[f'{split}_{column}'] for column in ('process_id','source_group','pm_cycle','cohort_key','window_start','window_end','position_ratio','position_bin')} for split in ('train','validation','test')}
encoder_d_models={int(meta['model_config']['d_model']) for _,meta in encoders.values()}
if len(encoder_d_models)!=1: raise ValueError(f'All STEP 1 encoders must share d_model, got {sorted(encoder_d_models)}')
ENCODER_D_MODEL=encoder_d_models.pop()
display(pd.DataFrame([{'step':s,**{f'{k}_windows':len(v[k]) for k in v},'train_cohorts':len(set(map(str,provenance[s]['train']['cohort_key'])))} for s,v in windows.items()]))

In [ ]:
class FdcComparisonPairs(Dataset):
    def __init__(self,array,meta,seed,limit=None,max_references=16,reference_array=None,reference_meta=None):
        self.x=array; self.meta=meta; self.seed=seed; self.max_references=max_references
        self.reference_array=array if reference_array is None else reference_array
        reference_meta=meta if reference_meta is None else reference_meta
        groups=defaultdict(list)
        for i,position in enumerate(reference_meta['position_bin']): groups[(int(position),str(reference_meta['cohort_key'][i]))].append(i)
        valid=[]; candidates={}
        for i,(position,process) in enumerate(zip(meta['position_bin'],meta['process_id'])):
            members={}
            for j in groups[(int(position),str(meta['cohort_key'][i]))]:
                run=str(reference_meta['process_id'][j])
                if run!=str(process): members.setdefault(run,j)
            pool=list(members.values())  # One reference window per independent TRAIN run.
            if pool: valid.append(i); candidates[i]=pool
        if limit and len(valid)>limit: valid=np.random.default_rng(seed).choice(valid,size=limit,replace=False).tolist()
        if not valid: raise ValueError('No cross-run pairs; check process and position provenance')
        self.indices=valid; self.candidates=candidates
    def __len__(self): return len(self.indices)
    def __getitem__(self,item):
        index=self.indices[item]; rng=np.random.default_rng(self.seed+item); pool=np.asarray(self.candidates[index]); count=min(self.max_references,len(pool)); selected=rng.choice(pool,size=count,replace=False); reference=np.median(self.reference_array[selected],axis=0).astype(np.float32)
        return torch.from_numpy(self.x[index]),torch.from_numpy(reference),int(self.meta['position_bin'][index])

def make_loaders(split,shuffle):
    result={}
    for i,(step,(_,ck)) in enumerate(encoders.items()):
        limit=CONFIG.max_windows_per_step if split=='train' else CONFIG.max_validation_windows_per_step
        ds=FdcComparisonPairs(windows[step][split],provenance[step][split],SEED+1000*i+{'train':0,'validation':100_000,'test':200_000}[split],limit,reference_array=windows[step]['train'],reference_meta=provenance[step]['train'])
        result[step]=DataLoader(ds,CONFIG.batch_size,shuffle=shuffle,num_workers=0)
    return result
train_loaders=make_loaders('train',True); validation_loaders=make_loaders('validation',False); test_loaders=make_loaders('test',False)
print({'train_pairs':sum(len(x.dataset) for x in train_loaders.values()),'validation_pairs':sum(len(x.dataset) for x in validation_loaders.values()),'test_pairs':sum(len(x.dataset) for x in test_loaders.values()),'protocol':PAIR_PROTOCOL})

In [ ]:
FDC_FEATURE_NAMES=('current_patch_mean','reference_patch_mean','absolute_mean_delta','current_patch_std','reference_patch_std','mean_absolute_point_delta','current_reconstruction_mse','reference_reconstruction_mse')
NUMERIC_DIM=4*ENCODER_D_MODEL+(56 if selected_variant()=='multiscale' else 24 if selected_variant()=='temporal' else len(FDC_FEATURE_NAMES))

def numeric_state(encoder,current,reference):
    from pecvd.temporal_bridge import supports, numeric_state as temporal_state
    if supports(encoder): return temporal_state(encoder,current,reference)
    with torch.no_grad():
        a=encoder.tokens(current); b=encoder.tokens(reference)
        current_patch,_=encoder.patchify(current); reference_patch,_=encoder.patchify(reference)
        current_reconstruction=encoder(current); reference_reconstruction=encoder(reference)
        fdc_features=torch.stack([current_patch.mean(-1),reference_patch.mean(-1),(current_patch.mean(-1)-reference_patch.mean(-1)).abs(),current_patch.std(-1,unbiased=False),reference_patch.std(-1,unbiased=False),(current_patch-reference_patch).abs().mean(-1),(current_reconstruction-current_patch).pow(2).mean(-1),(reference_reconstruction-reference_patch).pow(2).mean(-1)],-1)
    state=torch.cat([a,b,a-b,(a-b).abs(),fdc_features],-1)
    if state.shape[-1]!=NUMERIC_DIM: raise ValueError(f'Computed numeric state width {state.shape[-1]} != declared {NUMERIC_DIM}')
    return state

def numeric_token_descriptions(step,meta,patch_count,batch_size):
    names=meta.get('sensor_descriptors',{name:name for name in meta['sensor_columns']}); phase=meta.get('step_descriptor',step); one=gevis_backbone.token_descriptions(phase,[names.get(sensor,sensor) for sensor in meta['sensor_columns']],patch_count); return [one for _ in range(batch_size)]

SEMANTIC_NAMES=('mean level','variability','trend slope','local change','reconstruction error'); DIRECTION_NAMES=('increase','decrease','change'); SCOPE_NAMES=('whole','early','late')
def semantic_comparison_features(current,reference,state,patch_count):
    from pecvd.experiments.operations import semantic_comparison_features as shared_operation
    return shared_operation(current,reference,state,patch_count)

def grounded_question_contract(current, reference, state, position_bins, step, meta, sample_ids=None, wording_split='train'):
    from pecvd.experiments.questions import make_question
    batch, sensors = (len(current), current.shape[2])
    patch_count = state.shape[1] // sensors
    features = semantic_comparison_features(current, reference, state, patch_count)
    rows = torch.arange(batch, device=current.device)
    ids = rows if sample_ids is None else sample_ids.to(current.device)
    sensor_ids = (ids + position_bins.long()) % sensors
    scope_ids = (ids // 3 + 2 * position_bins.long()) % 3
    statistic_ids = (ids // 7 + position_bins.long()) % len(SEMANTIC_NAMES)
    direction_ids = (ids // 11 + position_bins.long()) % 3
    selected = features[rows, scope_ids, sensor_ids]
    values = selected[rows, statistic_ids]
    directed = torch.where(direction_ids.eq(0), values, torch.where(direction_ids.eq(1), -values, values.abs()))
    threshold = torch.tensor(1.0, device=current.device)
    boolean = torch.tanh(torch.stack([threshold - directed, directed - threshold], 1) / CONFIG.reward_temperature)
    strengths = selected.abs()
    strongest = strengths.amax(1)
    choice = torch.tanh(torch.cat([(2 * threshold - strongest)[:, None], strengths], 1) / CONFIG.reward_temperature)
    centers = torch.tensor([0.0, 1.0, 2.0, 3.0, 4.0], device=current.device)
    ordinal = torch.tanh(-(strongest[:, None] - centers[None]).abs() / CONFIG.reward_temperature)
    names = meta.get('sensor_descriptors', {name: name for name in meta['sensor_columns']})
    phase = meta.get('step_descriptor', step)
    sensor_text = [names.get(meta['sensor_columns'][int(index)], meta['sensor_columns'][int(index)]) for index in sensor_ids.cpu()]
    scope_text = [('throughout the full interval', 'during the early interval', 'during the late interval')[int(scope)] for scope in scope_ids.cpu()]
    boolean_q = [make_question(phase, sensor, 'BOOLEAN', ('level', 'variability', 'slope', 'local_change', 'residual')[int(stat)], ('increase', 'decrease', 'either')[int(direction)], ('whole', 'early', 'late')[int(scope)], split=wording_split, variant=int(stat)) for sensor, scope, stat, direction in zip(sensor_text, scope_ids.cpu(), statistic_ids.cpu(), direction_ids.cpu())]
    choice_q = [make_question(phase, sensor, 'CHOICE', scope=('whole', 'early', 'late')[int(scope)], split=wording_split) for sensor, scope in zip(sensor_text, scope_ids.cpu())]
    ordinal_q = [make_question(phase, sensor, 'ORDINAL', scope=('whole', 'early', 'late')[int(scope)], split=wording_split) for sensor, scope in zip(sensor_text, scope_ids.cpu())]
    return {'questions': {'BOOLEAN': boolean_q, 'CHOICE': choice_q, 'ORDINAL': ordinal_q}, 'rewards': {'BOOLEAN': boolean, 'CHOICE': choice, 'ORDINAL': ordinal}, 'sensor_ids': sensor_ids, 'scope_ids': scope_ids, 'statistic_ids': statistic_ids, 'direction_ids': direction_ids, 'features': features}


@torch.no_grad()
def prepare_fast_cache(model,loaders,split):
    model.eval(); result=[]; started=time.perf_counter(); total_bytes=0
    for step,loader in loaders.items():
        encoder,meta=encoders[step]; encoder=encoder.to(DEVICE).eval(); offset=0
        for batch_index,batch in enumerate(loader):
            current,reference,positions=[item.to(DEVICE) for item in batch]
            state=numeric_state(encoder,current,reference); pc=state.shape[1]//current.shape[2]
            ids=torch.arange(offset,offset+len(current),device=DEVICE); offset+=len(current)
            # Same fixed English questions and selected-center context as EXP4.
            from pecvd.answer_display import QUESTIONS
            from pecvd.experiments.questions import center_descriptions
            sensor_ids=(ids+positions.long())%current.shape[2]
            features=semantic_comparison_features(current,reference,state,pc)
            selected=features[torch.arange(len(current),device=DEVICE),0,sensor_ids].abs()
            strongest=selected.amax(-1); threshold=1.; temperature=CONFIG.reward_temperature
            rewards={'BOOLEAN':torch.tanh(torch.stack([threshold-strongest,strongest-threshold],1)/temperature),
                     'CHOICE':torch.tanh(torch.cat([(2*threshold-strongest)[:,None],selected],1)/temperature),
                     'ORDINAL':torch.tanh(-(strongest[:,None]-torch.arange(5,device=DEVICE)[None]).abs()/temperature)}
            descriptions=numeric_token_descriptions(step,meta,pc,len(current))
            descriptions=[center_descriptions(d,meta['sensor_columns'],pc,meta['sensor_columns'][int(i)]) for d,i in zip(descriptions,sensor_ids.cpu())]
            for answer_type in ANSWER_TYPES:
                fusion=model(state,[QUESTIONS[answer_type]]*len(current),[answer_type]*len(current),[CRITERIA[answer_type]]*len(current),descriptions,return_fusion=True)
                item={'step':step,'answer_type':answer_type,'fusion':tuple(t.detach().cpu() for t in fusion),'rewards':rewards[answer_type].cpu(),'sensor_ids':sensor_ids.cpu(),'patch_count':pc}
                total_bytes+=sum(t.numel()*t.element_size() for t in item['fusion'])+item['rewards'].numel()*4
                if total_bytes>CONFIG.max_cache_mib*1024**2: raise MemoryError('Fast cache budget exceeded; reduce max_windows_per_step')
                result.append(item)
            print({'phase':'prepare','split':split,'step':step,'batches_done':batch_index+1,'batches_total':len(loader),'elapsed_seconds':round(time.perf_counter()-started,2)},flush=True)
        encoder.cpu()
    return result

def macro_f1(y,p,n):
    scores=[]
    for cls in range(n):
        tp=np.sum((y==cls)&(p==cls)); fp=np.sum((y!=cls)&(p==cls)); fn=np.sum((y==cls)&(p!=cls))
        scores.append(2*tp/max(1,2*tp+fp+fn))
    return float(np.mean(scores))

@torch.no_grad()
def evaluate_cache(model,cache):
    model.eval(); collected={t:{'y':[],'p':[],'reward':[],'oracle':[]} for t in ANSWER_TYPES}; sensor_ok=[]
    for item in cache:
        fusion=tuple(t.to(DEVICE) for t in item['fusion']); t=item['answer_type']
        logits=torch.stack(model.decode(fusion,[t]*len(item['rewards']),[CRITERIA[t]]*len(item['rewards'])))
        pred=logits.argmax(-1).cpu(); rewards=item['rewards']; d=collected[t]
        d['y']+=rewards.argmax(-1).tolist(); d['p']+=pred.tolist()
        d['reward']+=rewards.gather(1,pred[:,None]).squeeze(1).tolist(); d['oracle']+=rewards.amax(-1).tolist()
        if t=='CHOICE':
            token=item['fusion'][2].argmax(-1); sensor_ok+=(token//item['patch_count']==item['sensor_ids']).tolist()
    metrics={'evidence_exact_accuracy':None,'evidence_sensor_accuracy':float(np.mean(sensor_ok)) if sensor_ok else None}
    for t,d in collected.items():
        y=np.asarray(d['y']); p=np.asarray(d['p']); present=np.unique(y)
        if not len(y): raise ValueError('Empty evaluation split')
        recalls=[float(np.mean(p[y==cls]==cls)) for cls in present]
        metrics[t]={'reward_argmax_accuracy':float(np.mean(y==p)),'balanced_accuracy':float(np.mean(recalls)),'macro_f1':macro_f1(y,p,len(CRITERIA[t])),'mean_selected_reward':float(np.mean(d['reward'])),'mean_reward_regret':float(np.mean(np.asarray(d['oracle'])-np.asarray(d['reward']))),'largest_prediction_share':max(Counter(p).values())/len(p),'target_largest_share':max(Counter(y).values())/len(y),'classes_present':len(present),'pairs':len(y)}
    return metrics

def score_metrics(m):
    return float(np.mean([m[t]['balanced_accuracy']+m[t]['macro_f1']-m[t]['mean_reward_regret'] for t in ANSWER_TYPES]))

def quality_gates(metrics):
    return {t:metrics[t]['classes_present']==len(CRITERIA[t]) and metrics[t]['balanced_accuracy']>=1/len(CRITERIA[t])+.05 and metrics[t]['mean_reward_regret']<=.20 for t in ANSWER_TYPES}



In [ ]:
seed_everything(); gevis=GEVIS(NUMERIC_DIM,CONFIG.d_model,CONFIG.dropout,CONFIG.text_encoder_name,CONFIG.text_encoder_revision).to(DEVICE)
backbone_path=BACKBONE_ROOT/'gevis_backbone.pt'
if not backbone_path.exists(): raise FileNotFoundError('Run STEP_2_train_gevis_backbone.ipynb first')
backbone_checkpoint=torch.load(backbone_path,map_location='cpu',weights_only=False)
if backbone_checkpoint.get('schema')!='gevis.backbone.generic.v10' or int(backbone_checkpoint.get('d_model',-1))!=CONFIG.d_model or int(backbone_checkpoint.get('numeric_dim',-1))!=NUMERIC_DIM: raise RuntimeError('STEP 2 generic-discovery backbone contract mismatch')
if backbone_checkpoint.get('step1_bridge',{}).get('manifest_sha256')!=sha256_file(manifest_path): raise RuntimeError('STEP 2 was not trained against this exact STEP 1 manifest')
from pecvd.encoder_loading import backbone_source_matches
if backbone_checkpoint.get('source_schema')!=BACKBONE_SOURCE_SCHEMA or not backbone_source_matches(backbone_checkpoint,Path(gevis_backbone.__file__)): raise RuntimeError('STEP2 source identity is incompatible; select a reviewed matching backbone')
if backbone_checkpoint.get('semantic_curriculum')!=BACKBONE_CURRICULUM_TAG or backbone_checkpoint.get('text_encoder_name')!=CONFIG.text_encoder_name or backbone_checkpoint.get('text_encoder_revision')!=CONFIG.text_encoder_revision: raise RuntimeError('STEP 2 E5 identity mismatch; run the updated STEP 2 experiment first')
backbone_state=backbone_checkpoint['backbone_state']; component=lambda name:{key.removeprefix(name+'.'):value for key,value in backbone_state.items() if key.startswith(name+'.')}
gevis.text.load_state_dict(component('text')); gevis.numeric_input_norm.load_state_dict(component('numeric_input_norm')); gevis.numeric.load_state_dict(component('numeric')); gevis.numeric_key_norm.load_state_dict(component('numeric_key_norm')); gevis.descriptor_norm.load_state_dict(component('descriptor_norm')); gevis.fusion_gate.load_state_dict(component('fusion_gate')); gevis.type_emb.load_state_dict(component('type_emb')); gevis.cross.load_state_dict(component('cross')); gevis.answer_norm.load_state_dict(component('norm'))
support_state={key.removeprefix('support_head.'):value for key,value in backbone_checkpoint['pretrain_heads_state'].items() if key.startswith('support_head.')}
gevis.support_head.load_state_dict(support_state)
gevis.fusion_mode=backbone_checkpoint.get('fusion_mode','legacy')
semantic_state={key.removeprefix('semantic_heads.'):value for key,value in backbone_checkpoint['pretrain_heads_state'].items() if key.startswith('semantic_heads.')}
gevis.semantic_heads.load_state_dict(semantic_state)

gevis.requires_grad_(False).eval()
print('Frozen STEP 1/2 loaded. Active answer system:',DECISION_SYSTEM)


# Answer decoder is retained only as an explicit optional ablation.
if RUN_ANSWER_HEAD_TRAINING:
    # Shared full-GEVIS artifact: no extra architecture, legacy search, or manual confirmation.
    importlib.reload(gevis_fast)
    CRITERIA={k:list(v) for k,v in gevis_fast.DEFAULT_ANSWER_CRITERIA.items()}  # Explicit new English answer-head contract, not display-only translation.
    ANSWER_CHECKPOINT_PATH=backbone_path.parent/'gevis_answer_model.pt'
    if ANSWER_CHECKPOINT_PATH.is_file():
        ANSWER_CHECKPOINT=gevis_fast.load_answer_checkpoint(gevis,ANSWER_CHECKPOINT_PATH,backbone_sha256=sha256_file(backbone_path),criteria=CRITERIA)
        print(f'Reused trained full GEVIS: {ANSWER_CHECKPOINT_PATH}')
    elif RUN_ANSWER_HEAD_TRAINING:
        train_runs={str(r) for step in provenance.values() for r in step['train']['process_id']}
        validation_runs={str(r) for step in provenance.values() for r in step['validation']['process_id']}
        if not train_runs or not validation_runs or train_runs & validation_runs: raise ValueError('TRAIN/Validation run overlap; do not fit answer heads')
        print('Preparing frozen encoder/fusion caches; TRAIN-only references, fixed English center questions.')
        ANSWER_TRAIN_CACHE=prepare_fast_cache(gevis,train_loaders,'train')
        ANSWER_VALIDATION_CACHE=prepare_fast_cache(gevis,validation_loaders,'validation')
        ANSWER_CHECKPOINT_PATH,ANSWER_HISTORY,ANSWER_TRAINING_REPORT=gevis_fast.fit_answer_checkpoint(
            gevis,ANSWER_TRAIN_CACHE,ANSWER_VALIDATION_CACHE,CRITERIA,backbone_path,
            train_run_ids=train_runs,validation_run_ids=validation_runs,epochs=CONFIG.epochs,
            batch_size=CONFIG.batch_size,learning_rate=CONFIG.learning_rate,
            max_seconds=CONFIG.max_fit_seconds,patience=CONFIG.patience,seed=SEED)
        ANSWER_VALIDATION_METRICS=evaluate_cache(gevis,ANSWER_VALIDATION_CACHE)
        display(pd.DataFrame({k:v for k,v in ANSWER_VALIDATION_METRICS.items() if k in ANSWER_TYPES}).T)
        print(f'Trained full GEVIS saved for EXP4: {ANSWER_CHECKPOINT_PATH}')
        print('Targets are numerical comparison proxies, NOT independent equipment-fault labels. Model softmax is uncalibrated.')
        del ANSWER_TRAIN_CACHE,ANSWER_VALIDATION_CACHE
    else:
        raise RuntimeError('Missing trained full GEVIS. Enable RUN_ANSWER_HEAD_TRAINING and run STEP3 from the top.')
    
else:
    print('Legacy decoder ablation disabled. Active answer system:',DECISION_SYSTEM)


In [ ]:
if globals().get('RUN_FUSION_ANSWER_TRAINING',False):
    from types import SimpleNamespace
    import pecvd.experiments.fusion_answers as fusion_answers
    fusion_answers=importlib.reload(fusion_answers)
    FUSION_RUNTIME=SimpleNamespace(module=SimpleNamespace(ROOT=ROOT,GEVIS_MODEL=gevis,
        STEP_ENCODERS=encoders,ENCODER_ROOT=ENCODER_ROOT,BACKBONE_PATH=backbone_path,
        DEVICE=DEVICE,numeric_state=numeric_state,numeric_token_descriptions=numeric_token_descriptions,
        RUN_COLUMN=RUN_COLUMN,STEP_COLUMN=STEP_COLUMN,ORDER_COLUMN=ORDER_COLUMN),encoders=encoders)
    FUSION_TRAINING=fusion_answers.prepare(FUSION_RUNTIME)
    print('Training recipe:', fusion_answers.TRAINING_RECIPE)
    print('Ordinal target:', fusion_answers.ORDINAL_TARGET)
    display(pd.DataFrame(FUSION_TRAINING['augmentation_coverage']))
    display(pd.DataFrame(FUSION_TRAINING['history']))
    FUSION_VALIDATION=fusion_answers.evaluate(FUSION_RUNTIME,split='validation')
    import pecvd.experiments.fusion_metrics as fusion_metrics
    fusion_metrics=importlib.reload(fusion_metrics)
    fusion_metrics.display_report(FUSION_VALIDATION,display)
    display(FUSION_VALIDATION.diagnostics['center_swap'])
    display(FUSION_VALIDATION.diagnostics['off_center'])
    display(FUSION_VALIDATION.diagnostics['intervention_metrics'])
    display(FUSION_VALIDATION.diagnostics['delay_metrics'])
    display(FUSION_VALIDATION.diagnostics['reconstruction_metrics'])
    print('Actual Fusion answer-head predictions. Validation selects epochs; final held-out answer evaluation is STEP 4.')
    print('Controlled paired perturbation targets, not verified fault truth. No numerical rule decides an answer.')
